# Structural Features Analysis: Do LLMs Have Different Fingerprints?

This notebook mirrors the original `structural_features_analysis (1).py` script but is organized into cells so you can:

- Inspect how structural features are loaded from `train.parquet`
- Run each analysis step interactively
- Easily extract textual statements for your slides (e.g., "AST max depth differs significantly across LLMs ...").


In [1]:
# Step 0: Setup
import pandas as pd
import numpy as np
from scipy.stats import kruskal, mannwhitneyu
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

print("Libraries loaded successfully")


Libraries loaded successfully


In [2]:
# Step 1: Load your data (update path if needed)
DATA_PATH = "../dataset/extract_structural_features/train.parquet"
SOURCE_COLUMN = "model"  # "model" = compare LLMs; switch to "source" for dataset origin (gh/lc/cf)

df = pd.read_parquet(DATA_PATH)
print(f"Data loaded from {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} samples, {df.shape[1]} columns")

print(f"\nGroups in '{SOURCE_COLUMN}' column:")
display(df[SOURCE_COLUMN].value_counts())

display(df.head())


Data loaded from ../dataset/extract_structural_features/train.parquet
Shape: 177,279 samples, 64 columns

Groups in 'model' column:


model
human        88828
codellama    22152
llama3.1     20516
qwen1.5      19910
nxcode       18394
gpt           7479
Name: count, dtype: int64

,language,model,split,target,source,dataset_split,split_index,sample_row_id,code_sha1,code_col_used,...,ast_count_GeneratorExp,ast_count_Lambda,hf_avgFunctionLength,hf_avgIdentifierLength,hf_avgLineLength,hf_emptyLinesDensity,hf_functionDefinitionDensity,hf_maintainabilityIndex,hf_maxDecisionTokens,hf_whiteSpaceRatio
0,python,human,train,human,gh,train,0,train:0,b70c1f89fbd370cead14155dc240ea014c9196e5,cleaned_code,...,0.0,0.0,1.0,3.985525,47.102041,0.040816,0.006803,0.0,0.0,0.405375
1,python,human,train,human,gh,train,1,train:1,e83fb11bc9adb44e004c34d9da1ea3f1d05a67cf,cleaned_code,...,0.0,0.0,1.0,5.000000,38.750000,0.000000,0.250000,0.0,0.0,0.272152
2,python,human,train,human,gh,train,2,train:2,d892a8ce1c2ad4e363c526c26061f2e833200d71,cleaned_code,...,0.0,0.0,1.0,5.210526,32.380952,0.142857,0.047619,0.0,0.0,0.340000
3,python,human,train,human,gh,train,3,train:3,b38c21b80b6d807455ab94b06f284103ae8ef7d1,cleaned_code,...,0.0,0.0,1.0,5.643678,30.791667,0.125000,0.020833,0.0,0.0,0.247869
4,python,human,train,human,gh,train,4,train:4,f4fa1c126b12663023b9a021847d404f383fc0f3,cleaned_code,...,0.0,0.0,1.0,5.291139,28.130435,0.260870,0.043478,0.0,0.0,0.270553


In [3]:
# Step 2: Data overview
print("Available columns:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")

print("\n--- Samples per MODEL (LLM / author) ---")
display(df["model"].value_counts())

print("\n--- Samples per SOURCE (dataset origin) ---")
display(df["source"].value_counts())

print(f"\nUsing SOURCE_COLUMN = '{SOURCE_COLUMN}' for the analysis.")

print("\nBasic statistics:")
display(df.describe().round(2))


Available columns:
  1. language
  2. model
  3. split
  4. target
  5. source
  6. dataset_split
  7. split_index
  8. sample_row_id
  9. code_sha1
  10. code_col_used
  11. char_count
  12. loc
  13. non_empty_loc
  14. comment_loc
  15. comment_density
  16. avg_line_length
  17. max_line_length
  18. token_count
  19. significant_token_count
  20. whitespace_ratio
  21. ast_parse_success
  22. ast_node_count
  23. ast_max_depth
  24. function_count
  25. class_count
  26. loop_count
  27. if_count
  28. try_count
  29. import_count
  30. call_count
  31. assign_count
  32. return_count
  33. comprehension_count
  34. cyclomatic_complexity
  35. ast_count_FunctionDef
  36. ast_count_AsyncFunctionDef
  37. ast_count_ClassDef
  38. ast_count_Return
  39. ast_count_Assign
  40. ast_count_AnnAssign
  41. ast_count_AugAssign
  42. ast_count_For
  43. ast_count_AsyncFor
  44. ast_count_While
  45. ast_count_If
  46. ast_count_Try
  47. ast_count_With
  48. ast_count_AsyncWith
  49. ast_co

model
human        88828
codellama    22152
llama3.1     20516
qwen1.5      19910
nxcode       18394
gpt           7479
Name: count, dtype: int64


--- Samples per SOURCE (dataset origin) ---


source
gh    79895
lc    58737
cf    38647
Name: count, dtype: int64


Using SOURCE_COLUMN = 'model' for the analysis.

Basic statistics:


,split_index,char_count,loc,non_empty_loc,comment_loc,comment_density,avg_line_length,max_line_length,token_count,significant_token_count,...,ast_count_GeneratorExp,ast_count_Lambda,hf_avgFunctionLength,hf_avgIdentifierLength,hf_avgLineLength,hf_emptyLinesDensity,hf_functionDefinitionDensity,hf_maintainabilityIndex,hf_maxDecisionTokens,hf_whiteSpaceRatio
count,177279.00,177279.00,177279.00,177279.00,177279.0,177279.0,177279.00,177279.00,177279.00,177279.00,...,177279.00,177279.00,177279.00,177279.00,177279.00,177279.00,177279.00,177279.00,177279.00,177279.00
mean,88639.00,642.12,32.22,17.78,0.0,0.0,27.36,67.28,186.25,142.96,...,0.06,0.19,0.76,4.69,30.59,0.18,0.08,12.94,0.20,0.29
std,51176.18,1227.10,83.80,33.32,0.0,0.0,119.19,216.20,386.12,291.33,...,0.33,0.93,0.43,1.82,62.34,0.19,0.12,20.94,1.49,0.10
min,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00
25%,44319.50,183.00,7.00,5.00,0.0,0.0,17.75,44.00,51.00,39.00,...,0.00,0.00,1.00,3.57,21.57,0.00,0.01,0.00,0.00,0.22
50%,88639.00,342.00,13.00,10.00,0.0,0.0,25.80,61.00,93.00,73.00,...,0.00,0.00,1.00,4.73,29.80,0.14,0.05,0.00,0.00,0.28
75%,132958.50,663.00,27.00,19.00,0.0,0.0,33.82,78.00,185.00,146.00,...,0.00,0.00,1.00,5.63,37.80,0.25,0.10,21.93,0.00,0.35
max,177278.00,57512.00,4123.00,1514.00,0.0,0.0,49421.00,49421.00,30299.00,30081.00,...,26.00,51.00,1.00,230.28,24710.50,0.96,1.00,100.00,105.00,0.84


In [4]:
# Step 3: Select structural features — all names match actual columns in train.parquet
STRUCTURAL_FEATURES = [
    # AST structure
    "ast_max_depth",
    "ast_node_count",
    # Complexity
    "cyclomatic_complexity",
    # Code constructs
    "function_count",
    "class_count",
    "loop_count",
    "if_count",
    "try_count",
    "import_count",
    "call_count",
    "assign_count",
    "return_count",
    "comprehension_count",
    # Size & style
    "loc",
    "avg_line_length",
    "whitespace_ratio",
    # HalsteadFan / maintainability metrics
    "hf_avgFunctionLength",
    "hf_avgIdentifierLength",
    "hf_emptyLinesDensity",
    "hf_functionDefinitionDensity",
    "hf_maintainabilityIndex",
    "hf_maxDecisionTokens",
]

available_features = [f for f in STRUCTURAL_FEATURES if f in df.columns]
missing_features   = [f for f in STRUCTURAL_FEATURES if f not in df.columns]

if missing_features:
    print("Warning: features not found in data:", missing_features)
print(f"Analyzing {len(available_features)} features: {available_features}")


Analyzing 22 features: ['ast_max_depth', 'ast_node_count', 'cyclomatic_complexity', 'function_count', 'class_count', 'loop_count', 'if_count', 'try_count', 'import_count', 'call_count', 'assign_count', 'return_count', 'comprehension_count', 'loc', 'avg_line_length', 'whitespace_ratio', 'hf_avgFunctionLength', 'hf_avgIdentifierLength', 'hf_emptyLinesDensity', 'hf_functionDefinitionDensity', 'hf_maintainabilityIndex', 'hf_maxDecisionTokens']


In [5]:
# Step 4: Statistical tests (Kruskal-Wallis) and summaries
def run_statistical_tests(df, features, source_col):
    results = []
    sources = df[source_col].unique()
    for feature in features:
        groups = [df[df[source_col] == s][feature].dropna().values for s in sources]
        h_stat, p_value = kruskal(*groups)
        n = sum(len(g) for g in groups)
        k = len(groups)
        eta_squared = (h_stat - k + 1) / (n - k)
        if p_value < 0.001:
            significance = "*** (p < 0.001)"
        elif p_value < 0.01:
            significance = "** (p < 0.01)"
        elif p_value < 0.05:
            significance = "* (p < 0.05)"
        else:
            significance = "ns (not significant)"
        if eta_squared >= 0.14:
            effect = "Large"
        elif eta_squared >= 0.06:
            effect = "Medium"
        elif eta_squared >= 0.01:
            effect = "Small"
        else:
            effect = "Negligible"
        results.append({
            "Feature": feature,
            "H-statistic": round(h_stat, 2),
            "p-value": p_value,
            "Significance": significance,
            "Effect Size (eta_sq)": round(eta_squared, 4),
            "Effect Interpretation": effect,
        })
    return pd.DataFrame(results)

def pairwise_comparisons(df, feature, source_col, alpha=0.05):
    sources = df[source_col].unique()
    pairs = list(combinations(sources, 2))
    adjusted_alpha = alpha / len(pairs)
    rows = []
    for s1, s2 in pairs:
        g1 = df[df[source_col] == s1][feature].dropna().values
        g2 = df[df[source_col] == s2][feature].dropna().values
        stat, p_val = mannwhitneyu(g1, g2, alternative="two-sided")
        med1, med2 = np.median(g1), np.median(g2)
        rows.append({
            "Pair": f"{s1} vs {s2}",
            "Median 1": round(med1, 2),
            "Median 2": round(med2, 2),
            "p-value": p_val,
            "Significant (Bonferroni)": "Yes" if p_val < adjusted_alpha else "No",
        })
    return pd.DataFrame(rows)

def summary_by_source(df, features, source_col):
    summary = df.groupby(source_col)[features].agg(["mean", "median", "std"])
    summary.columns = ["_".join(col).strip() for col in summary.columns.values]
    return summary.round(2)

results_df = run_statistical_tests(df, available_features, SOURCE_COLUMN)
display(results_df)

significant_features = results_df[results_df["p-value"] < 0.05]["Feature"].tolist()
print("Significant features:", significant_features)

summary_df = summary_by_source(df, available_features, SOURCE_COLUMN)
display(summary_df)


,Feature,H-statistic,p-value,Significance,Effect Size (eta_sq),Effect Interpretation
0,ast_max_depth,28767.69,0.000000e+00,*** (p < 0.001),0.1623,Large
1,ast_node_count,34308.55,0.000000e+00,*** (p < 0.001),0.1935,Large
2,cyclomatic_complexity,30096.49,0.000000e+00,*** (p < 0.001),0.1697,Large
3,function_count,4173.53,0.000000e+00,*** (p < 0.001),0.0235,Small
4,class_count,1532.12,0.000000e+00,*** (p < 0.001),0.0086,Negligible
5,loop_count,10942.53,0.000000e+00,*** (p < 0.001),0.0617,Medium
6,if_count,27543.62,0.000000e+00,*** (p < 0.001),0.1553,Large
7,try_count,4374.44,0.000000e+00,*** (p < 0.001),0.0246,Small
8,import_count,1953.88,0.000000e+00,*** (p < 0.001),0.0110,Small
9,call_count,38706.16,0.000000e+00,*** (p < 0.001),0.2183,Large


Significant features: ['ast_max_depth', 'ast_node_count', 'cyclomatic_complexity', 'function_count', 'class_count', 'loop_count', 'if_count', 'try_count', 'import_count', 'call_count', 'assign_count', 'return_count', 'comprehension_count', 'loc', 'avg_line_length', 'whitespace_ratio', 'hf_avgFunctionLength', 'hf_avgIdentifierLength', 'hf_emptyLinesDensity', 'hf_functionDefinitionDensity', 'hf_maintainabilityIndex', 'hf_maxDecisionTokens']


,ast_max_depth_mean,ast_max_depth_median,ast_max_depth_std,ast_node_count_mean,ast_node_count_median,ast_node_count_std,cyclomatic_complexity_mean,cyclomatic_complexity_median,cyclomatic_complexity_std,function_count_mean,...,hf_emptyLinesDensity_std,hf_functionDefinitionDensity_mean,hf_functionDefinitionDensity_median,hf_functionDefinitionDensity_std,hf_maintainabilityIndex_mean,hf_maintainabilityIndex_median,hf_maintainabilityIndex_std,hf_maxDecisionTokens_mean,hf_maxDecisionTokens_median,hf_maxDecisionTokens_std
model,,,,,,,,,,,,,,,,,,,,,
codellama,6.91,7.0,2.92,60.04,41.0,65.38,2.70,2.0,2.73,0.93,...,0.10,0.16,0.11,0.15,17.69,2.87,23.23,0.03,0.0,0.64
gpt,8.47,9.0,2.31,123.93,83.0,110.84,4.34,3.0,3.49,1.15,...,0.08,0.07,0.05,0.06,7.99,0.00,14.61,0.10,0.0,1.16
human,9.15,9.0,2.44,235.33,119.0,407.40,7.33,4.0,10.61,1.82,...,0.22,0.05,0.03,0.05,12.40,0.00,21.17,0.36,0.0,1.93
llama3.1,7.45,8.0,2.70,74.13,57.0,64.84,2.68,2.0,2.45,0.96,...,0.09,0.09,0.08,0.09,14.67,0.29,20.24,0.04,0.0,0.71
nxcode,6.05,7.0,3.68,62.84,46.0,68.02,2.42,2.0,2.59,0.78,...,0.11,0.12,0.07,0.18,11.96,0.00,21.08,0.06,0.0,0.82
qwen1.5,6.47,7.0,3.46,66.75,49.0,75.37,2.65,2.0,2.88,0.78,...,0.11,0.12,0.07,0.17,11.05,0.00,18.66,0.06,0.0,0.87


In [6]:
# Step 5: Natural-language summary statements for each significant feature

# Human-readable labels for each feature
FEATURE_LABELS = {
    "ast_max_depth":              "AST max depth",
    "ast_node_count":             "AST node count",
    "cyclomatic_complexity":      "cyclomatic complexity",
    "function_count":             "number of functions",
    "class_count":                "number of classes",
    "loop_count":                 "loop count",
    "if_count":                   "number of conditionals",
    "try_count":                  "exception-handling block count",
    "import_count":               "import count",
    "call_count":                 "function call count",
    "assign_count":               "assignment count",
    "return_count":               "return statement count",
    "comprehension_count":        "comprehension count",
    "loc":                        "lines of code",
    "avg_line_length":            "average line length",
    "whitespace_ratio":           "whitespace ratio",
    "hf_avgFunctionLength":       "average function length",
    "hf_avgIdentifierLength":     "average identifier length",
    "hf_emptyLinesDensity":       "empty-line density",
    "hf_functionDefinitionDensity": "function-definition density",
    "hf_maintainabilityIndex":    "maintainability index",
    "hf_maxDecisionTokens":       "max decision tokens",
}

def format_p(p):
    if p < 0.001:
        return "p < 0.001"
    elif p < 0.01:
        return "p < 0.01"
    elif p < 0.05:
        return "p < 0.05"
    return f"p = {p:.3f}"

def feature_sentence(df, results_df, feature, source_col=SOURCE_COLUMN):
    if feature not in results_df["Feature"].values:
        return f"Feature '{feature}' not found in results."

    row     = results_df[results_df["Feature"] == feature].iloc[0]
    p_str   = format_p(row["p-value"])
    effect  = row["Effect Interpretation"].lower()
    label   = FEATURE_LABELS.get(feature, feature.replace("_", " "))

    medians = df.groupby(source_col)[feature].median().sort_values(ascending=False)
    top     = medians.index[0]
    top_med = medians.iloc[0]
    others  = [(medians.index[i], medians.iloc[i]) for i in range(1, len(medians))]

    # Sentence 1: significance
    sent1 = (f"{label.capitalize()} differs significantly across LLMs "
             f"(Kruskal-Wallis {p_str}, {effect} effect).")

    # Sentence 2: ranked medians
    others_str = " and ".join(
        [f"{name} (median={med:.2f})" for name, med in others]
    ) if others else ""
    sent2 = f"{top} produces the highest {label} (median={top_med:.2f})"
    if others_str:
        sent2 += f" compared to {others_str}."
    else:
        sent2 += "."

    return f"{sent1} {sent2}"


print("=" * 80)
print("DISTRIBUTION ANALYSIS — TEXTUAL STATEMENTS")
print("=" * 80)
for feat in significant_features:
    print()
    print(feature_sentence(df, results_df, feat))
print()
print("=" * 80)
print(f"Total significant features (α=0.05): {len(significant_features)} / {len(available_features)}")


DISTRIBUTION ANALYSIS — TEXTUAL STATEMENTS

Ast max depth differs significantly across LLMs (Kruskal-Wallis p < 0.001, large effect). gpt produces the highest AST max depth (median=9.00) compared to human (median=9.00) and llama3.1 (median=8.00) and codellama (median=7.00) and nxcode (median=7.00) and qwen1.5 (median=7.00).

Ast node count differs significantly across LLMs (Kruskal-Wallis p < 0.001, large effect). human produces the highest AST node count (median=119.00) compared to gpt (median=83.00) and llama3.1 (median=57.00) and qwen1.5 (median=49.00) and nxcode (median=46.00) and codellama (median=41.00).

Cyclomatic complexity differs significantly across LLMs (Kruskal-Wallis p < 0.001, large effect). human produces the highest cyclomatic complexity (median=4.00) compared to gpt (median=3.00) and codellama (median=2.00) and llama3.1 (median=2.00) and nxcode (median=2.00) and qwen1.5 (median=2.00).

Number of functions differs significantly across LLMs (Kruskal-Wallis p < 0.001, s

Comprehension count differs significantly across LLMs (Kruskal-Wallis p < 0.001, small effect). codellama produces the highest comprehension count (median=0.00) compared to gpt (median=0.00) and human (median=0.00) and llama3.1 (median=0.00) and nxcode (median=0.00) and qwen1.5 (median=0.00).

Lines of code differs significantly across LLMs (Kruskal-Wallis p < 0.001, large effect). human produces the highest lines of code (median=21.00) compared to gpt (median=16.00) and qwen1.5 (median=10.00) and nxcode (median=10.00) and llama3.1 (median=9.00) and codellama (median=7.00).

Average line length differs significantly across LLMs (Kruskal-Wallis p < 0.001, negligible effect). human produces the highest average line length (median=27.89) compared to llama3.1 (median=25.08) and codellama (median=25.00) and qwen1.5 (median=24.88) and nxcode (median=24.30) and gpt (median=24.12).

Whitespace ratio differs significantly across LLMs (Kruskal-Wallis p < 0.001, medium effect). human produces the